In [22]:
import torch
import torch.nn.functional as F

from tqdm import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from dataclasses import dataclass

In [23]:
IGNORE_INDEX=-100
MAX_LENGTH=1024

In [24]:
from dataclasses import dataclass


@dataclass
class Config:
    pretrained_lm: str = "openai-community/gpt2"
    ift_dataset: str = "HuggingFaceTB/smoltalk"
    # the pretrained lm does not come with a chat template
    chat_template_tokenizer: str = "HuggingFaceTB/SmolLM2-135M-Instruct"
    dataset_subset: str = "everyday-conversations"
    dataset_split: str = "train"
    max_length: int = MAX_LENGTH

    # training hyperparams
    lr: float = 5.0e-6
    num_epochs: int = 3
    batch_size: int = 4
    gradient_accumulation_steps: int = 8
    warmup_ratio: float = 0.1
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    seed: int = 42

    # Hardware
    bf16: bool = True
    gradient_checkpointing: bool = True
    model_device_id: int = 0

    # Logging
    wandb_project: str | None = None
    wandb_run_name: str | None = None


In [25]:
def load_model(cfg, device: torch.device):
    tokenizer = AutoTokenizer.from_pretrained(
        cfg.pretrained_lm, trust_remote_code=False
    )

    if tokenizer.chat_template is None and cfg.chat_template_tokenizer:
        donor = AutoTokenizer.from_pretrained(
            cfg.chat_template_tokenizer, trust_remote_code=False
        )
        if donor.chat_template is None:
            raise ValueError(
                f"chat_template_source {cfg.chat_template_tokenizer} has no chat_template."
            )
        tokenizer.chat_template = donor.chat_template

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if cfg.bf16 else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        cfg.pretrained_lm,
        trust_remote_code=False,
        dtype=dtype,
    ).to(device)

    return model, tokenizer


In [26]:
model, tokenizer = load_model(Config, "mps")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [27]:
# function to test model output
tokenizer = AutoTokenizer.from_pretrained(Config.pretrained_lm)
donor = AutoTokenizer.from_pretrained(Config.chat_template_tokenizer)

tokenizer.chat_template = donor.chat_template

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

messages = [
    {"role": "user", "content": "Hello how are you?"}
]


inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
    )


response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>assistant
Hello, I'm SmolLM, trained by 

In [28]:
# for Seb Raschka's ift dataset
import os
import requests
import json

def download_and_load_file(file_path, url):
    # Download file only if it does not exist (idempotent)
    if not os.path.exists(file_path):
        r = requests.get(url)
        r.raise_for_status()
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(r.text)
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

file_path = "instruction-data.json"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
seb_data = download_and_load_file(file_path, url)

def transform_to_messages_wrapped(data):
    """
    Transforms a list of dicts with keys 'instruction', 'input', and 'output'
    into a list of dicts with a "messages" key, where each value is a list of multiturn
    conversation messages (dicts with 'role' and 'content').
    This allows downstream code to access example["messages"] as expected.
    """
    examples = []
    for example in data:
        conversation = []
        if example.get("instruction"):
            conversation.append({"role": "user", "content": example["instruction"]})
        if example.get("input"):
            conversation.append({"role": "user", "content": example["input"]})
        if example.get("output"):
            conversation.append({"role": "assistant", "content": example["output"]})
        if conversation:  # Only add non-empty conversations
            examples.append({"messages": conversation})
    return examples

seb_dataset = transform_to_messages_wrapped(seb_data)
seb_dataset

[{'messages': [{'role': 'user',
    'content': 'Evaluate the following phrase by transforming it into the spelling given.'},
   {'role': 'user', 'content': 'freind --> friend'},
   {'role': 'assistant',
    'content': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}]},
 {'messages': [{'role': 'user',
    'content': 'Edit the following sentence for grammar.'},
   {'role': 'user', 'content': 'He go to the park every day.'},
   {'role': 'assistant', 'content': 'He goes to the park every day.'}]},
 {'messages': [{'role': 'user', 'content': 'Convert 45 kilometers to meters.'},
   {'role': 'assistant', 'content': '45 kilometers is 45000 meters.'}]},
 {'messages': [{'role': 'user',
    'content': "Rewrite this sentence to start with 'Although': Despite the rain, they went for a walk."},
   {'role': 'assistant',
    'content': 'Although it was raining, they went for a walk.'}]},
 {'messages': [{'role': 'user',
    'content': 'What are the first 10 sq

In [29]:
def encode_example(example, tokenizer: AutoTokenizer, max_length):
    messages = example["messages"]

    # apply_chat_template(messages) != apply_chat_template(messages[-1:]) + apply_chat_template(messages[:-1])
    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True, return_dict=False
    )

    full_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False, return_dict=False
    )

    assert full_ids[:len(prompt_ids)] == prompt_ids

    if len(full_ids) > max_length:
        return None

    labels = [IGNORE_INDEX] * len(prompt_ids)
    labels += full_ids[len(prompt_ids) :]

    input_ids = full_ids[:max_length]
    labels = labels[:max_length]

    if all(x == IGNORE_INDEX for x in labels):
        return None

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

@dataclass
class SFTBatch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    labels: torch.Tensor

    def to(self, device: torch.device | str) -> "SFTBatch":
        return SFTBatch(
            input_ids=self.input_ids.to(device),
            attention_mask=self.attention_mask.to(device),
            labels=self.labels.to(device),
        )


class SFTDataset(Dataset):
    def __init__(self, raw_dataset, tokenizer, max_length):
        self.examples = []

        for row in raw_dataset:
            encoded = encode_example(row, tokenizer, max_length)
            if encoded is not None:
                self.examples.append(encoded)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collator(examples, pad_token_id):
    max_len = max(ex["input_ids"].size(0) for ex in examples)

    input_ids = []
    attention_mask = []
    labels = []

    for ex in examples:
        ids = ex["input_ids"]
        lbl = ex["labels"]
        pad_len = max_len - ids.size(0)

        input_ids.append(
            torch.cat(
                [
                    ids,
                    torch.full(
                        size=(pad_len,), fill_value=pad_token_id, dtype=torch.long
                    ),
                ]
            )
        )

        attention_mask.append(
            torch.cat(
                [
                    torch.ones(size=(ids.size(0),), dtype=torch.long),
                    torch.zeros(size=(pad_len,), dtype=torch.long),
                ]
            )
        )

        labels.append(
            torch.cat(
                [
                    lbl,
                    torch.full(
                        size=(pad_len,), fill_value=IGNORE_INDEX, dtype=torch.long
                    ),
                ]
            )
        )

    return SFTBatch(
        input_ids=torch.stack(input_ids),
        attention_mask=torch.stack(attention_mask),
        labels=torch.stack(labels),
    )


def create_dataloader(cfg, tokenizer: AutoTokenizer):

    # raw_dataset = load_dataset(cfg.ift_dataset, cfg.dataset_subset, split=cfg.dataset_split)
    raw_dataset = seb_dataset

    ds = SFTDataset(raw_dataset, tokenizer, max_length=MAX_LENGTH)

    pad_token_id = tokenizer.pad_token_id

    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=lambda batch: collator(batch, pad_token_id),
        num_workers=0,
        pin_memory=False,
    )


In [30]:
def compute_loss(model, batch):
    output = model(
        input_ids=batch.input_ids,
        attention_mask=batch.attention_mask,
        use_cache=False,
    )
    # shift for next token prediction
    shift_logits = output.logits[:, :-1, :].contiguous()
    shift_labels = batch.labels[:, 1:].contiguous()
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=IGNORE_INDEX,
    )

In [31]:
cfg = Config
device = "mps"

model, tokenizer = load_model(cfg, device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [32]:
model, tokenizer

(GPT2LMHeadModel(
   (transformer): GPT2Model(
     (wte): Embedding(50257, 768)
     (wpe): Embedding(1024, 768)
     (drop): Dropout(p=0.1, inplace=False)
     (h): ModuleList(
       (0-11): 12 x GPT2Block(
         (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
         (attn): GPT2Attention(
           (c_attn): Conv1D(nf=2304, nx=768)
           (c_proj): Conv1D(nf=768, nx=768)
           (attn_dropout): Dropout(p=0.1, inplace=False)
           (resid_dropout): Dropout(p=0.1, inplace=False)
         )
         (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
         (mlp): GPT2MLP(
           (c_fc): Conv1D(nf=3072, nx=768)
           (c_proj): Conv1D(nf=768, nx=3072)
           (act): NewGELUActivation()
           (dropout): Dropout(p=0.1, inplace=False)
         )
       )
     )
     (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
   )
   (lm_head): Linear(in_features=768, out_features=50257, bias=Fal

In [33]:
dataloader = create_dataloader(cfg, tokenizer)

In [34]:
len(dataloader)

275

In [35]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)

In [36]:
model.train()
optimizer.zero_grad(set_to_none=True)

In [44]:
for epoch in range(cfg.num_epochs):
    print(f"Epoch {epoch + 1}/{cfg.num_epochs}:")
    with tqdm(dataloader, desc=f"Epoch {epoch+1}/{cfg.num_epochs}") as pbar:
        for batch in pbar:
            batch = batch.to(device)
            loss = compute_loss(model, batch)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            pbar.set_postfix({"loss": f"{loss:.4f}"})
   

Epoch 1/3:


Epoch 1/3: 100%|██████████| 275/275 [00:38<00:00,  7.21it/s, loss=1.1953]


Epoch 2/3:


Epoch 2/3: 100%|██████████| 275/275 [00:36<00:00,  7.62it/s, loss=1.8516]


Epoch 3/3:


Epoch 3/3: 100%|██████████| 275/275 [00:36<00:00,  7.61it/s, loss=1.6328]


In [45]:
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [46]:
# function to test model output
tokenizer = AutoTokenizer.from_pretrained(cfg.pretrained_lm)
donor = AutoTokenizer.from_pretrained(cfg.chat_template_tokenizer)

tokenizer.chat_template = donor.chat_template

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

messages = [
    {"role": "user", "content": "what's up?"}
]


inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<|im_end|>
<|im_start|>user
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im

In [40]:
print(
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Hello how are you?<|im_end|>
<|im_start|>assistant

